# 03_daily_flows.ipynb
## Агрегация дневных денежных потоков

In [1]:
import pandas as pd
import numpy as np
import os

print("=== Cash Flow at Risk - Daily Aggregation ===\n")

=== Cash Flow at Risk - Daily Aggregation ===



In [2]:
# Путь к очищенному файлу
CSV_CLEAN = "transactions_clean.csv"

# Если файла нет, создаём тестовые данные
if not os.path.exists(CSV_CLEAN):
    print("⚠️ Файл transactions_clean.csv не найден!")
    print("   Создаю тестовые данные для отладки...\n")
    
    np.random.seed(42)
    test_data = pd.DataFrame({
        'trans_date': pd.date_range('2024-01-01', periods=365, freq='D'),
        'amount': np.random.normal(1000, 500, 365),
        'trans_type': np.random.choice(['inflow', 'outflow'], 365, p=[0.6, 0.4])
    })
    # Добавляем несколько отрицательных дней для реалистичности
    test_data.loc[50:60, 'amount'] = -2000
    test_data.loc[50:60, 'trans_type'] = 'outflow'
    
    test_data.to_csv(CSV_CLEAN, index=False)
    print(f"✅ Создан тестовый файл: {CSV_CLEAN}\n")

⚠️ Файл transactions_clean.csv не найден!
   Создаю тестовые данные для отладки...

✅ Создан тестовый файл: transactions_clean.csv



In [3]:
# Загружаем данные
df = pd.read_csv(CSV_CLEAN)
df['trans_date'] = pd.to_datetime(df['trans_date']).dt.date

# Агрегация по дням
daily = df.groupby('trans_date').apply(
    lambda x: pd.Series({
        'inflow': x[x['trans_type'] == 'inflow']['amount'].sum(),
        'outflow': x[x['trans_type'] == 'outflow']['amount'].sum(),
        'net_flow': x[x['trans_type'] == 'inflow']['amount'].sum() - x[x['trans_type'] == 'outflow']['amount'].sum(),
        'txn_count': len(x)
    })
).reset_index()

daily.columns = ['day', 'inflow', 'outflow', 'net_flow', 'txn_count']

# Сохраняем
daily.to_csv("daily_cashflow.csv", index=False)

print(f"📊 Результат агрегации:")
print(f"   - Период: {daily['day'].min()} → {daily['day'].max()}")
print(f"   - Всего дней: {len(daily)}")
print(f"   - Суммарный приток: {daily['inflow'].sum():,.2f}")
print(f"   - Суммарный отток: {daily['outflow'].sum():,.2f}")
print(f"   - Чистый поток: {daily['net_flow'].sum():,.2f}")

print(f"\n✅ Сохранено в daily_cashflow.csv")
print("\n📋 Первые 5 дней:")
print(daily.head())

📊 Результат агрегации:
   - Период: 2024-01-01 → 2024-12-30
   - Всего дней: 365
   - Суммарный приток: 211,885.84
   - Суммарный отток: 121,171.76
   - Чистый поток: 90,714.08

✅ Сохранено в daily_cashflow.csv

📋 Первые 5 дней:
          day       inflow      outflow     net_flow  txn_count
0  2024-01-01  1248.357077     0.000000  1248.357077        1.0
1  2024-01-02     0.000000   930.867849  -930.867849        1.0
2  2024-01-03  1323.844269     0.000000  1323.844269        1.0
3  2024-01-04     0.000000  1761.514928 -1761.514928        1.0
4  2024-01-05   882.923313     0.000000   882.923313        1.0


C:\Users\korephan\AppData\Local\Temp\ipykernel_26156\522153123.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily = df.groupby('trans_date').apply(
